# Same-model adaptation and independent replication

Reproduce `experiments/v3_protocol.json`: Whisper-base LoRA, stricter five-recording WavLM enrollment, and frozen intent models evaluated on unused en-AU/en-GB MInDS-14 recordings.

Cells are unexecuted. Equivalent CLI commands produce the saved evidence. Allow at least 15 GB free disk and substantial CPU time. No paid services are used. These are research samples, not official benchmark scores.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
repo = Path('/content/speech-v3') if Path('/content').exists() else Path.cwd() / 'speech-v3'
if repo.exists(): raise RuntimeError('Choose a fresh checkout directory')
subprocess.run(['git','clone','--branch','codex/siri-real-speech-lab-20261119','https://github.com/ByteBoss123/Speech-Command-.git',str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','torch==2.8.0','--index-url','https://download.pytorch.org/whl/cpu'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r','requirements-speech.txt'],check=True)
def run(module, *arguments):
    subprocess.run([sys.executable,'-m',module,*arguments],check=True)
print('Code commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())

## Reconstruct exclusions and prepare fresh data

Earlier manifests identify all previously used speakers/utterances. Prior evaluation examples are excluded from new evaluation.

In [ ]:
run('speechlab.prepare','--revision','71cacbfb7e2354c4226d01e70d77d5fca3d04ba1')
run('speechlab.prepare_v2')
run('speechlab.intent_data')
run('speechlab.intent_data_v2')
run('speechlab.prepare_v3')

## Whisper-base LoRA

Both arms use the same pinned 72.6M-parameter backbone. The unchanged baseline competes with three adapter epochs on generated development WER. Fresh test probes are scored only after selection is saved.

In [ ]:
run('speechlab.asr_v3')
print(Path('runs/asr-v3/metrics.json').read_text())

## Speaker operating point

Compare frozen v2 one/three-recording thresholds against five-recording enrollment. Select the new threshold/method on development at an empirical 0.5% FAR cap. This is not a guaranteed population bound.

In [ ]:
run('speechlab.speaker_v3')
print(Path('runs/speaker-v3/metrics.json').read_text())

## Independent intent replication

No weights or blend are changed on this external dataset. All prior IDs, normalized transcripts and audio hashes are excluded. English-variety replication does not validate new French/Spanish performance or speaker independence.

In [ ]:
run('speechlab.intent_external')
print(Path('runs/intent-external-v3/metrics.json').read_text())

## Separate extension: all nine remaining languages

The English-variety replication did not establish a reliable gain. This extension evaluates every remaining language without changing models or stopping early. Report both comparisons separately with Bonferroni-adjusted primary p-values; nominal bootstrap intervals remain descriptive.

In [ ]:
run('speechlab.intent_external', '--data','data/minds14-unseen-v3', '--output','runs/intent-unseen-v3', '--languages','cs-CZ','de-DE','it-IT','ko-KR','nl-NL','pl-PL','pt-PT','ru-RU','zh-CN', '--previous','data/minds14-pilot/manifest.jsonl','data/minds14-v2/manifest.jsonl','data/minds14-external-v3/manifest.jsonl', '--protocol','experiments/v3_multilingual_extension_protocol.json')
print(Path('runs/intent-unseen-v3/metrics.json').read_text())

## Export and verify this run

The allowlist excludes audio, transcripts, source IDs and individual embeddings. Hardware may cause small numerical differences.

In [ ]:
run('pytest','tests_speech','-q')
run('speechlab.bundle_v3','--output','runs/reproduced-evidence-v3')
run('speechlab.verify_v3','runs/reproduced-evidence-v3')